In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit.Chem import MolFromSmiles
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import train_test_split
from pathlib import Path
from sklearn.model_selection import train_test_split
from lightning import pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from sklearn.model_selection import KFold
from chemprop import data, featurizers, models, nn
from rdkit.Chem import MolFromSmiles
import concurrent.futures
import logging
from chemprop.data import datapoints, dataloader, MoleculeDataset
from chemprop.featurizers import SimpleMoleculeMolGraphFeaturizer
from chemprop.data.datapoints import MoleculeDatapoint
import torch
import chemprop.nn.metrics as chem_metrics
from chemprop.nn.agg import (
    MultiHeadAttentiveAggregation,
    GatedAttentiveAggregation,
    GatedAttentiveAggregationv2
)
from assets.functionchem import *
from assets.chempropcombination import *

df = pd.read_csv("./data/Fupdata.csv")

# Apply the scaffold computation for each SMILES
df['scaffold'] = df['smiles'].apply(compute_scaffold)

# Step 1: Get the unique scaffolds
scaffold_list = df['scaffold'].unique()

# Step 2: Split the list of scaffolds into train, test, and validation
train_scaffolds, temp_scaffolds = train_test_split(scaffold_list, test_size=0.2, random_state=42)
test_scaffolds, val_scaffolds = train_test_split(temp_scaffolds, test_size=0.5, random_state=42)

# Step 3: Create the train, test, and validation sets by filtering the original DataFrame based on scaffold
train_df = df[df['scaffold'].isin(train_scaffolds)]
test_df = df[df['scaffold'].isin(test_scaffolds)]
val_df = df[df['scaffold'].isin(val_scaffolds)]

# Step 4: Verify the distribution of scaffolds in each set
print(f"Training set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}")
print(f"Validation set size: {len(val_df)}")

results_df = run_chemprop_mp_agg_benchmark(
    train_df,
    val_df,
    test_df,
    target_column="Fu_final",
    max_epochs=200
)

Seed set to 42


Training set size: 3740
Test set size: 467
Validation set size: 468

Training: BondMP3 + Mean


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
You are using a CUDA device ('NVIDIA GeForce RTX 4070 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing │  227 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation    │      0 │ train │     0 │
│ 2 │ bn              │ BatchNorm1d        │    600 │ train │     0 │
│ 3 │ predictor       │ RegressionFFN      │ 90.6 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity           │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList         │      0 │ train │     0 │
└───┴─────────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 318 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 318 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 26                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

Output()

`Trainer.fit` stopped: `max_epochs=200` reached.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

Test: {'Target': 'lgFu', 'MessagePassing': 'AtomMP6', 'Aggregation': 'MultiHead12', 'Test_MAE': 0.3627831041812897, 'Test_RMSE': 0.49786049127578735, 'Test_R2': 0.6088541746139526}

Final comparison:
   Target MessagePassing     Aggregation  Test_MAE  Test_RMSE   Test_R2
16   lgFu        BondMP5      MultiHead8  0.336122   0.461489  0.663918
13   lgFu        BondMP5  GatedAttentive  0.344533   0.459235  0.667193
18   lgFu        BondMP6            Mean  0.344937   0.469904  0.651549
23   lgFu        BondMP6     MultiHead12  0.345313   0.469736  0.651798
17   lgFu        BondMP5     MultiHead12  0.346864   0.473442  0.646282
14   lgFu        BondMP5      Attentive1  0.347180   0.476700  0.641397
15   lgFu        BondMP5      MultiHead4  0.347474   0.477067  0.640845
7    lgFu        BondMP4  GatedAttentive  0.348717   0.480358  0.635873
20   lgFu        BondMP6      Attentive1  0.349039   0.472886  0.647112
12   lgFu        BondMP5            Mean  0.352149   0.479874  0.636607
22   lgF